# STPLS3D Dataset Preparation Pipeline (Colab)

This notebook prepares dataset splits (`train/val/test`) for training:
1. `PLY -> prepared LAS` (stretch to multiples of 250, density cap, reports)
2. `prepared LAS tiles -> tensors + PNG` (`img_features`, `img_rgb`, `img_class`)

Notes:
- Designed for Colab runtime.
- Paths are configured in Windows style strings, then normalized to POSIX for Colab execution.
- Each split is launched manually (`train`, `val`, `test`) and errors are collected into a summary table.



In [ ]:
# Optional dependencies (run once per fresh Colab runtime)
# !pip install -q -r /content/lidar-visual-interface/requirements.txt

In [1]:
from __future__ import annotations

import os
import sys
import shlex
import subprocess
from pathlib import Path, PureWindowsPath
from dataclasses import dataclass, asdict

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / 'training').exists() and (PROJECT_ROOT / 'notebooks').exists():
    # Keep as-is for repo-root runs; adjust manually only for custom layouts.
    pass
print('PROJECT_ROOT =', PROJECT_ROOT)
print('PYTHON      =', sys.executable)
print('exists      =', PROJECT_ROOT.exists())

# Ensure local packages are importable when running scripts/modules
os.environ['PYTHONPATH'] = f"{PROJECT_ROOT}:{os.environ.get('PYTHONPATH','')}"

PROJECT_ROOT = c:\Users\alexe\VSCprojects\lidar-visual-interface
PYTHON      = c:\Users\alexe\VSCprojects\lidar-visual-interface\venv\Scripts\python.exe
exists      = True


In [22]:
# -------- Configuration --------
# Keep Windows-style paths here; they are converted to Colab POSIX paths by to_colab_path().

CFG = {
    # Repository root in Windows style (for documentation consistency)
    'repo_root_win': PROJECT_ROOT,

    # Input PLY split directories (Windows style placeholders)
    'ply_root_win': r'D:\data\STPLS3D_ply\STPLS3D_ply',

    # Output roots (Windows style placeholders)
    'prepared_root_win': r'D:\data\STPLS3D_ply\STPLS3D_ply\stpls3d_prepared',
    'ready_root_win': r'D:\data\STPLS3D_ply\STPLS3D_ply\stpls3d_ready',

    # Default parameters (project defaults)
    'tile_size': 250,
    'target_points_per_tile': 2_000_000,
    'seed': 42,
    'num_points': 30_000,
    'grid_size': 512,
    'k_nn': 4,
    'knn_eps': 0.0,
    'knn_workers': 1,
}

# Stage toggles
RUN_PLY_TO_LAS = True
RUN_PREPARE_DATA = True

# Split selector (run one split at a time for manual control)
TARGET_SPLIT = 'train'  # 'train' | 'val' | 'test'

CFG



{'repo_root_win': WindowsPath('c:/Users/alexe/VSCprojects/lidar-visual-interface'),
 'ply_root_win': 'D:\\data\\STPLS3D_ply\\STPLS3D_ply',
 'prepared_root_win': 'D:\\data\\STPLS3D_ply\\STPLS3D_ply\\stpls3d_prepared',
 'ready_root_win': 'D:\\data\\STPLS3D_ply\\STPLS3D_ply\\stpls3d_ready',
 'tile_size': 250,
 'target_points_per_tile': 2000000,
 'seed': 42,
 'num_points': 30000,
 'grid_size': 512,
 'k_nn': 4,
 'knn_eps': 0.0,
 'knn_workers': 1}

In [10]:
from pathlib import Path, PureWindowsPath
import os

def to_runtime_path(path_str: str) -> Path:
    # Local Windows: keep drive paths as-is
    if os.name == "nt":
        return Path(PureWindowsPath(path_str))
    # Colab/Linux: interpret only POSIX paths directly
    p = Path(path_str)
    return p if p.is_absolute() else (PROJECT_ROOT / p)

def to_colab_path(win_path: str, repo_root: Path = PROJECT_ROOT) -> Path:
    # Map Windows-style repo-local path into Colab path under PROJECT_ROOT.
    p = PureWindowsPath(win_path)
    parts = list(p.parts)
    key = 'lidar-visual-interface'
    if key in parts:
        idx = parts.index(key)
        rel = Path(*parts[idx + 1:])
        return repo_root / rel
    # Fallback: interpret as relative-ish path
    return repo_root / Path(*[x for x in parts if x not in (p.anchor, '\\')])


def split_paths(split: str) -> dict:
    assert split in {'train', 'val', 'test'}

    ply_root = to_runtime_path(CFG['ply_root_win'])
    ply_dir = ply_root / split

    prepared_root = to_runtime_path(CFG['prepared_root_win'])
    ready_root = to_runtime_path(CFG['ready_root_win'])

    prepared_split = prepared_root / split
    ready_split = ready_root / split

    return {
        'ply_dir': ply_dir,
        'prepared_split': prepared_split,
        'tiles_las_dir': prepared_split / 'tiles_las',
        'ready_split': ready_split,
        'features_dir': ready_split / 'img_features',
        'rgb_dir': ready_split / 'img_rgb',
        'class_dir': ready_split / 'img_class',
        'tensor_dir': ready_split / 'tensors',
        'reports_dir': prepared_split / 'reports',
    }


@dataclass
class StepResult:
    split: str
    stage: str
    cmd: str
    returncode: int
    ok: bool
    note: str


RESULTS: list[StepResult] = []


def run_cmd(args: list[str], cwd: Path = PROJECT_ROOT, check: bool = False) -> subprocess.CompletedProcess:
    cmd = [str(a) for a in args]
    print('\n$ ' + ' '.join(shlex.quote(c) for c in cmd))
    proc = subprocess.run(cmd, cwd=str(cwd), text=True, capture_output=True)
    if proc.stdout.strip():
        print(proc.stdout)
    if proc.stderr.strip():
        print(proc.stderr)
    if check and proc.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {proc.returncode}')
    return proc


def count_files(path: Path, pattern: str) -> int:
    if not path.exists():
        return 0
    return len(list(path.glob(pattern)))



## 1) Resolve current split paths and quick checks


In [23]:
paths = split_paths(TARGET_SPLIT)
for k, v in paths.items():
    print(f"{k:15s}: {v}")

print('\nInput PLY exists:', paths['ply_dir'].exists())
print('Existing LAS tiles:', count_files(paths['tiles_las_dir'], '*.las'))



ply_dir        : D:\data\STPLS3D_ply\STPLS3D_ply\train
prepared_split : D:\data\STPLS3D_ply\STPLS3D_ply\stpls3d_prepared\train
tiles_las_dir  : D:\data\STPLS3D_ply\STPLS3D_ply\stpls3d_prepared\train\tiles_las
ready_split    : D:\data\STPLS3D_ply\STPLS3D_ply\stpls3d_ready\train
features_dir   : D:\data\STPLS3D_ply\STPLS3D_ply\stpls3d_ready\train\img_features
rgb_dir        : D:\data\STPLS3D_ply\STPLS3D_ply\stpls3d_ready\train\img_rgb
class_dir      : D:\data\STPLS3D_ply\STPLS3D_ply\stpls3d_ready\train\img_class
tensor_dir     : D:\data\STPLS3D_ply\STPLS3D_ply\stpls3d_ready\train\tensors
reports_dir    : D:\data\STPLS3D_ply\STPLS3D_ply\stpls3d_prepared\train\reports

Input PLY exists: True
Existing LAS tiles: 0


## 2) Stage A ? PLY -> Prepared LAS (optional per split)

Runs `training/prepare_stpls3d_las.py` and keeps going even on partial failures.



In [24]:
if RUN_PLY_TO_LAS:
    proc = run_cmd([
        sys.executable,
        'training/prepare_stpls3d_las.py',
        '--input-dir', paths['ply_dir'],
        '--output-dir', paths['prepared_split'],
        '--tile-size', CFG['tile_size'],
        '--target-points-per-tile', CFG['target_points_per_tile'],
        '--seed', CFG['seed'],
    ])

    # Script convention: 0 = all ok, 2 = completed with per-file errors
    ok = proc.returncode in (0, 2)
    RESULTS.append(StepResult(
        split=TARGET_SPLIT,
        stage='ply_to_las',
        cmd='prepare_stpls3d_las.py',
        returncode=proc.returncode,
        ok=ok,
        note='0=ok, 2=partial file errors'
    ))
else:
    print('RUN_PLY_TO_LAS=False -> skipped')




$ 'c:\Users\alexe\VSCprojects\lidar-visual-interface\venv\Scripts\python.exe' training/prepare_stpls3d_las.py --input-dir 'D:\data\STPLS3D_ply\STPLS3D_ply\train' --output-dir 'D:\data\STPLS3D_ply\STPLS3D_ply\stpls3d_prepared\train' --tile-size 250 --target-points-per-tile 2000000 --seed 42
22:16:45 [INFO] Starting STPLS3D LAS preparation
22:16:45 [INFO] Input dir:  D:\data\STPLS3D_ply\STPLS3D_ply\train
22:16:45 [INFO] Output dir: D:\data\STPLS3D_ply\STPLS3D_ply\stpls3d_prepared\train
22:18:11 [INFO] Processed 10_points_GTv2.ply
22:19:19 [INFO] Processed 10_points_GTv3.ply
22:20:20 [INFO] Processed 11_points_GTv3.ply
22:21:14 [INFO] Processed 12_points_GTv3.ply
22:22:06 [INFO] Processed 13_points_GTv3.ply
22:23:21 [INFO] Processed 14_points_GTv3.ply
22:24:36 [INFO] Processed 15_points_GTv3.ply
22:25:49 [INFO] Processed 16_points_GTv3.ply
22:26:52 [INFO] Processed 17_points_GTv3.ply
22:28:06 [INFO] Processed 18_points_GTv3.ply
22:29:18 [INFO] Processed 19_points_GTv3.ply
22:30:26 [INFO]

## 3) Stage B ? LAS tiles -> tensors + PNG (optional per split)

Runs `training.prepare_data` with `--skip-tiling` to avoid double tiling artifacts.



In [25]:
if RUN_PREPARE_DATA:
    proc = run_cmd([
        sys.executable,
        '-m',
        'training.prepare_data',
        '--raw-dir', paths['tiles_las_dir'],
        '--output-dir', paths['ready_split'],
        '--skip-tiling',
        '--num-points', CFG['num_points'],
        '--grid-size', CFG['grid_size'],
        '--k-nn', CFG['k_nn'],
        '--knn-eps', CFG['knn_eps'],
        '--knn-workers', CFG['knn_workers'],
    ])

    RESULTS.append(StepResult(
        split=TARGET_SPLIT,
        stage='prepare_data',
        cmd='-m training.prepare_data',
        returncode=proc.returncode,
        ok=(proc.returncode == 0),
        note='expects pre-tiled LAS in --raw-dir'
    ))
else:
    print('RUN_PREPARE_DATA=False -> skipped')




$ 'c:\Users\alexe\VSCprojects\lidar-visual-interface\venv\Scripts\python.exe' -m training.prepare_data --raw-dir 'D:\data\STPLS3D_ply\STPLS3D_ply\stpls3d_prepared\train\tiles_las' --output-dir 'D:\data\STPLS3D_ply\STPLS3D_ply\stpls3d_ready\train' --skip-tiling --num-points 30000 --grid-size 512 --k-nn 4 --knn-eps 0.0 --knn-workers 1
08:34:48 [INFO] Step 1/5: Skipping tiling, using pre-tiled LAS from D:\data\STPLS3D_ply\STPLS3D_ply\stpls3d_prepared\train\tiles_las
08:34:48 [INFO] Step 2/5: Encoding to feature tensors...
08:34:52 [INFO] Encoding 230 LAS files to tensors
08:34:55 [INFO] Encoding 10_points_GTv2_0_0.las
08:34:58 [INFO] Encoding 10_points_GTv2_0_250.las
08:35:01 [INFO] Encoding 10_points_GTv2_250_0.las
08:35:04 [INFO] Encoding 10_points_GTv2_250_250.las
08:35:07 [INFO] Encoding 10_points_GTv3_0_0.las
08:35:10 [INFO] Encoding 10_points_GTv3_0_250.las
08:35:13 [INFO] Encoding 10_points_GTv3_250_0.las
08:35:16 [INFO] Encoding 10_points_GTv3_250_250.las
08:35:19 [INFO] Encoding

## 4) Validation checks for current split


In [7]:
checks = {
    'tiles_las': count_files(paths['tiles_las_dir'], '*.las'),
    'tensors_npy': count_files(paths['tensor_dir'], '*.npy'),
    'img_features_png': count_files(paths['features_dir'], '*.png'),
    'img_rgb_png': count_files(paths['rgb_dir'], '*.png'),
    'img_class_png': count_files(paths['class_dir'], '*.png'),
}
print(pd.Series(checks))

# Consistency checks
n_feat = checks['img_features_png']
n_rgb = checks['img_rgb_png']
n_cls = checks['img_class_png']
print('\nPNG count consistency (features/rgb/class):', n_feat, n_rgb, n_cls)
if not (n_feat == n_rgb == n_cls):
    print('WARNING: PNG counts mismatch across output folders.')



tiles_las           0
tensors_npy         0
img_features_png    0
img_rgb_png         0
img_class_png       0
dtype: int64

PNG count consistency (features/rgb/class): 0 0 0


## 5) Optional: run all splits sequentially (continue on errors)


In [ ]:
def run_split(split: str):
    global TARGET_SPLIT
    TARGET_SPLIT = split
    p = split_paths(split)

    if RUN_PLY_TO_LAS:
        proc_a = run_cmd([
            sys.executable,
            'training/prepare_stpls3d_las.py',
            '--input-dir', p['ply_dir'],
            '--output-dir', p['prepared_split'],
            '--tile-size', CFG['tile_size'],
            '--target-points-per-tile', CFG['target_points_per_tile'],
            '--seed', CFG['seed'],
        ])
        RESULTS.append(StepResult(split, 'ply_to_las', 'prepare_stpls3d_las.py', proc_a.returncode, proc_a.returncode in (0, 2), '0=ok,2=partial'))

    if RUN_PREPARE_DATA:
        proc_b = run_cmd([
            sys.executable,
            '-m', 'training.prepare_data',
            '--raw-dir', p['tiles_las_dir'],
            '--output-dir', p['ready_split'],
            '--skip-tiling',
            '--num-points', CFG['num_points'],
            '--grid-size', CFG['grid_size'],
            '--k-nn', CFG['k_nn'],
            '--knn-eps', CFG['knn_eps'],
            '--knn-workers', CFG['knn_workers'],
        ])
        RESULTS.append(StepResult(split, 'prepare_data', '-m training.prepare_data', proc_b.returncode, proc_b.returncode == 0, ''))


# Uncomment to run all splits in one go:
# for _split in ['train', 'val', 'test']:
#     print(f"\n===== RUN SPLIT: {_split} =====")
#     run_split(_split)



## 6) Error summary (across executed steps)


In [ ]:
if RESULTS:
    df = pd.DataFrame([asdict(x) for x in RESULTS])
    display(df)
    errors = df[~df['ok']]
    print('\nFailed steps:', len(errors))
    if len(errors):
        display(errors)
else:
    print('No steps executed yet.')

